# Backpropagation — Learning representations by back-propagating errors (1986)

_Papers · Foundations & Optimization_

**Compute the gradient of the error with respect to every weight in one backward sweep by the chain rule — so a network can learn its own internal (hidden) features.**

---

This notebook is a practice scaffold for the **Backpropagation — Learning representations by back-propagating errors (1986)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import math, torch

# =========================================================
#  Reverse-mode autodiff FROM SCRATCH — a tiny 'tape'.
#  Each Value records the op that made it + how to push grad
#  to its parents (the chain rule). Rumelhart et al. (1986).
# =========================================================
class Value:
    def __init__(self, data, parents=()):
        self.data = float(data)
        self.grad = 0.0                       # dE/d(this)
        self._parents = parents               # the Values this was built from
        self._backward = lambda: None         # how to push grad to parents

    def __add__(self, o):                     # x_j = sum_i y_i w_ji  (eq 1) uses +
        o = o if isinstance(o, Value) else Value(o)
        out = Value(self.data + o.data, (self, o))
        def bw():
            self.grad += out.grad             # d(a+b)/da = 1
            o.grad    += out.grad             # accumulate! (eq 7 is a SUM)
        out._backward = bw; return out

    def __mul__(self, o):                     # ... and *  (eq 1)
        o = o if isinstance(o, Value) else Value(o)
        out = Value(self.data * o.data, (self, o))
        def bw():
            self.grad += o.data * out.grad    # chain rule: d(ab)/da = b
            o.grad    += self.data * out.grad
        out._backward = bw; return out

    def sigmoid(self):                        # y_j = 1/(1+e^-x)  (eq 2)
        s = 1.0 / (1.0 + math.exp(-self.data))
        out = Value(s, (self,))
        def bw():
            self.grad += s * (1 - s) * out.grad   # eq (5): derivative is y(1-y)
        out._backward = bw; return out

    def __pow__(self, p):                     # for the 0.5*(y-d)^2 error (eq 3)
        out = Value(self.data ** p, (self,))
        def bw():
            self.grad += p * self.data ** (p - 1) * out.grad
        out._backward = bw; return out

    def __radd__(self, o): return self + o
    def __rmul__(self, o): return self * o

    def backward(self):
        # topological order (parents before children), then replay in REVERSE
        topo, seen = [], set()
        def build(v):
            if id(v) not in seen:
                seen.add(id(v))
                for p in v._parents: build(p)
                topo.append(v)
        build(self)
        self.grad = 1.0                       # dE/dE = 1 (start the backward pass)
        for v in reversed(topo):
            v._backward()                     # apply eqs (4)-(7) layer by layer

# ---------- a 2-layer MLP: 3 inputs -> 2 hidden (sigmoid) -> 1 output (sigmoid) ----------
torch.manual_seed(0)
W1 = torch.randn(2, 3, dtype=torch.float64); b1 = torch.randn(2, dtype=torch.float64)
W2 = torch.randn(1, 2, dtype=torch.float64); b2 = torch.randn(1, dtype=torch.float64)
x  = torch.randn(3, dtype=torch.float64)
d  = 1.0                                       # desired output

# --- my tape ---
xv  = [Value(float(v)) for v in x]
W1v = [[Value(float(W1[i, j])) for j in range(3)] for i in range(2)]
b1v = [Value(float(b1[i])) for i in range(2)]
W2v = [Value(float(W2[0, j])) for j in range(2)]
b2v = Value(float(b2[0]))

h = []
for i in range(2):                             # forward pass (eq 1,2)
    z = b1v[i]
    for j in range(3): z = z + W1v[i][j] * xv[j]
    h.append(z.sigmoid())
zo = b2v
for j in range(2): zo = zo + W2v[j] * h[j]
yo = zo.sigmoid()
E = 0.5 * (yo + (-1.0) * Value(d)) ** 2         # total error (eq 3)
E.backward()                                    # the backward pass
my_gW1 = torch.tensor([[W1v[i][j].grad for j in range(3)] for i in range(2)],
                      dtype=torch.float64)

# --- THE ORACLE: torch.autograd computes the same gradient ---
W1t = W1.clone().requires_grad_(); b1t = b1.clone().requires_grad_()
W2t = W2.clone().requires_grad_(); b2t = b2.clone().requires_grad_()
ht  = torch.sigmoid(W1t @ x + b1t)
yot = torch.sigmoid(W2t @ ht + b2t)
Et  = 0.5 * (yot - d) ** 2
Et.backward()

print("E  (mine) =", round(E.data, 8), " (torch) =", round(Et.item(), 8))
print("dE/dW1 allclose vs torch.autograd:",
      torch.allclose(my_gW1, W1t.grad, atol=1e-9))   # expect True  <-- the proof

# ---------- recompute the worked example: 1 -> 1 -> 1 net, d=0 ----------
yi = Value(0.5); w_hi = Value(0.2); w_oh = Value(0.3)
y_h = (yi * w_hi).sigmoid()
y_o = (y_h * w_oh).sigmoid()
Ee  = 0.5 * (y_o + (-0.0)) ** 2
Ee.backward()
print("worked example  dE/dw_oh =", round(w_oh.grad, 6),   # ~ 0.070342
      " dE/dw_hi =", round(w_hi.grad, 6))                  # ~ 0.005012

## Visualize the data & results

_Use the paper's equations (1)-(8), coded by hand, to train a tiny 2-hidden-layer net on XOR — a task that PROVABLY needs hidden units. Does the error fall and the network learn the right function?_

In [ ]:
import numpy as np
# Train a 2-4-1 sigmoid net on XOR using ONLY the paper's eqs (1)-(8), coded by hand.
rng = np.random.default_rng(0)
X = np.array([[0,0],[0,1],[1,0],[1,1]], float)
d = np.array([0,1,1,0], float)                     # XOR target (needs a hidden layer)
H = 4
W1 = rng.normal(0,1,(H,2)); b1 = rng.normal(0,1,H)
W2 = rng.normal(0,1,H);     b2 = rng.normal(0,1)
sig = lambda z: 1/(1+np.exp(-z))
eps = 0.5
log = []
for t in range(2001):
    xh = X @ W1.T + b1; yh = sig(xh)               # forward  (eq 1,2)
    xo = yh @ W2 + b2;  yo = sig(xo)
    E = 0.5*np.sum((yo-d)**2)                       # error    (eq 3)
    if t % 100 == 0: log.append((t, round(float(E),4)))
    dyo = yo - d                                    # eq 4
    dxo = dyo*yo*(1-yo)                             # eq 5  (delta_o)
    dW2 = dxo @ yh; db2 = dxo.sum()                 # eq 6
    dyh = np.outer(dxo, W2)                         # eq 7  (propagate down)
    dxh = dyh*yh*(1-yh)                             # eq 5  (delta_h)
    dW1 = dxh.T @ X; db1 = dxh.sum(0)               # eq 6
    W1-=eps*dW1; b1-=eps*db1; W2-=eps*dW2; b2-=eps*db2   # eq 8
print(log)                                          # E: 0.84 -> 0.003
print("final preds:", np.round(sig((sig(X@W1.T+b1))@W2+b2),3))  # ~ [0.034,0.959,0.960,0.045]

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** One input unit $i$, one output unit $o$ (no hidden layer), sigmoid, target $d=1$. Weight $w_{oi}=0.5$, input $y_i=2$. Do a forward pass and compute $\partial E/\partial w_{oi}$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Forward (eq 1-2): $x_o=y_i w_{oi}=2\cdot 0.5=1.0$; $y_o=1/(1+e^{-1})=0.731059$. — _Weighted sum, then squash._
- Error sensitivity (eq 4): $\partial E/\partial y_o=y_o-d=0.731059-1=-0.268941$. — _Only the target enters here._
- Delta (eq 5): $\partial E/\partial x_o=-0.268941\cdot y_o(1-y_o)=-0.268941\cdot 0.196612=-0.052878$. — _Push through the sigmoid; $y(1-y)$ is its derivative._
- Weight gradient (eq 6): $\partial E/\partial w_{oi}=-0.052878\cdot y_i=-0.052878\cdot 2=-0.105756$. — _Delta times the activity on the wire._

**Answer:** $\partial E/\partial w_{oi}\approx -0.1058$. It is negative, so gradient descent ($\Delta w=-\varepsilon\,\partial E/\partial w$) will INCREASE $w_{oi}$ — pushing the output up toward the target $d=1$, as expected.

</details>

**Problem 2.** Take the worked example (input $\to$ hidden $\to$ output, $\partial E/\partial w_{hi}=0.005012$). With learning rate $\varepsilon=0.1$ (eq 8), what is the new $w_{hi}$, and does the update reduce the output?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Apply eq (8): $\Delta w_{hi}=-\varepsilon\,\partial E/\partial w_{hi}=-0.1\cdot 0.005012=-0.0005012$. — _Step downhill on the error surface._
- New weight: $w_{hi}=0.2-0.0005012=0.1994988$. — _The gradient was positive, so the weight decreases._
- Reason about direction: target was $d=0$ but $y_o=0.539$ was too high; decreasing $w_{hi}$ lowers $x_h$, then $y_h$, then $x_o$, then $y_o$. — _Backprop credited this buried weight correctly._

**Answer:** $w_{hi}\approx 0.19950$. Because the output ($0.539$) was above the target ($0$), backprop produced a positive gradient, so eq (8) shrinks $w_{hi}$, nudging the output down toward 0. The hidden-layer weight was trained with no direct target — that is the paper's whole point.

</details>

**Problem 3.** Ablation: in the CODE's tape, replace the topological-order backward with a single pass that visits nodes in the order they were created (forward order), instead of reverse. What breaks, and why?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall a node's grad is only final once ALL its children have pushed into it (eq 7 is a sum over children). — _Reverse-mode needs children processed before parents._
- Visiting in forward order means a parent's $\_backward$ runs before its children have contributed, so it reads $grad=0$ and pushes zero (or partial) gradient down. — _The chain-rule product is broken mid-graph._
- Compare to torch.autograd: the allclose check now returns False. — _The oracle exposes the bug immediately._

**Answer:** Lower-layer gradients come out wrong (often near zero), so $\partial E/\partial w$ for the first-layer weights is incorrect and torch.allclose returns False. Reverse-mode autodiff is correct only when nodes are processed in REVERSE topological order — children fully before parents — because eq (7) sums contributions from all higher units, which must already be computed.

</details>